# Installing Tools

In [ ]:
#Installing plink2.0
!git clone https://github.com/chrchang/plink-ng.git
%cd plink-ng
%cd 2.0
!make
!sudo mv /content/plink-ng/2.0/bin/plink2 /usr/local/bin/
%cd ../..


In [ ]:
#Installing flashpca
!wget https://github.com/gabraham/flashpca/releases/download/v2.0/flashpca_x86-64.gz
!gunzip flashpca_x86-64.gz
!chmod +x flashpca_x86-64
!mv flashpca_x86-64 /usr/local/bin/flashpca


# Downloading Files

In [ ]:
import gdown

#Downloading vcf file
file_id = "1yuEscQ2lVxO_FhVHd1bf4QIlOP_79Slr"
url = f"https://drive.google.com/uc?id={file_id}"
output = "ALL.chr1.filtered_Pass.vcf.gz"
gdown.download(url, output, quiet=False)

#Downloading prune file
file_id = "130QTi8b5lljeCoH-p8ZlCgSb_lmbYw4y"
url = f"https://drive.google.com/uc?id={file_id}"
output = "pruned.prune.in"
gdown.download(url, output, quiet=False)

#Downloading pca mean sd file
file_id = "1-Lrf01Lk0BqQz6wmeZ3VSNGtRaqhFwYP"
url = f"https://drive.google.com/uc?id={file_id}"
output = "pca_meansd.txt"
gdown.download(url, output, quiet=False)

#Downloading pca loadings file
file_id = "1-GmnpocvHfW2cSWv9K5zP4kE3ni5wz3L"
url = f"https://drive.google.com/uc?id={file_id}"
output = "pca_loadings.txt"
gdown.download(url, output, quiet=False)

#Downloading knn_model file
file_id = "1PFkhd03a7NhTAc8ZzcHpsf8ZZ3glzlv0"
url = f"https://drive.google.com/uc?id={file_id}"
output = "knn_model.pkl"
gdown.download(url, output, quiet=False)

#Downloading centroids file
file_id = "1--ss25NC7XZeMGZxzY-r2MQbOBiLoGkn"
url = f"https://drive.google.com/uc?id={file_id}"
output = "centroids.npy"
gdown.download(url, output, quiet=False)

#Downloading scaler file
file_id = "1-0OOKZhDbGk5-x46SP62Q8vaidzLT_FI"
url = f"https://drive.google.com/uc?id={file_id}"
output = "scaler.pkl"
gdown.download(url, output, quiet=False)



# Preparing a Test Data

In [ ]:
!plink2 \
  --vcf ALL.chr1.filtered_Pass.vcf.gz \
  --make-bed \
  --out test_data

In [ ]:
!plink2 --bfile  test_data --keep <(echo "HG03398") --recode vcf --out test_data

# Testing

In [13]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import time


In [14]:
import joblib

# Load the KNN model
knn = joblib.load('knn_model.pkl')

# Load the centroids
centroids = np.load('centroids.npy')

# Load the scaler
scaler = joblib.load('scaler.pkl')

In [ ]:
start_time = time.time()

!plink2 \
  --vcf test_data.vcf \
  --make-bed \
  --out test_data

!plink2 \
  --bfile test_data \
  --set-all-var-ids '@#:$1:$2' \
  --new-id-max-allele-len 120 truncate \
  --make-bed \
  --out test_data_unique_ids

!plink2 \
  --bfile test_data_unique_ids \
  --extract pruned.prune.in \
  --make-bed \
  --out test_data_pruned

!flashpca --bfile test_data_pruned --ndim 4 --project --inload pca_loadings.txt --inmeansd pca_meansd.txt --outproj test_data_projected.txt

X_test = pd.read_csv("test_data_projected.txt", delim_whitespace=True)[['PC1', 'PC2', 'PC3', 'PC4']].copy()
X_test_scaled = scaler.transform(X_test)

distances = cdist(X_test_scaled, centroids, metric='euclidean')
inverse_distances = 1 / (distances + 1e-10)
probabilities = inverse_distances / inverse_distances.sum()

# probabilities = knn.predict_proba(X_test_scaled)

elapsed_time = time.time() - start_time

In [19]:
print(probabilities * 100)


[[ 7.5877677  75.84358218  8.04626105  8.52238907]]


In [20]:
print("Elapsed time: {:.2f} seconds".format(elapsed_time))


Elapsed time: 1.33 seconds
